<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_00_raw_data_ingestion/stage_00_raw_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_00_raw_data_preparation**

## Resumen

Esta notebook realiza la **preparación inicial del dataset intradía del MNQ (Micro E-mini Nasdaq 100)**.  
El objetivo es construir un dataset limpio, consistente y estructurado que servirá como base para la ingeniería de factores y el entrenamiento de modelos.

0. **Configuración del entorno**
   - Clonado del repositorio y montaje de Google Drive.
   - Instalación e importación de librerías necesarias.

1. **Fuente de datos**
   - Datos históricos intradía del MNQ (OHLCV, frecuencia de 1 minuto) exportados desde NinjaTrader.
   - Archivos originales en formato `.txt`, en zona horaria UTC.

2. **Generación del dataset**
   - Unificación de todos los archivos `.txt` en un único DataFrame.
   - Asignación de nombres de columnas: `open`, `high`, `low`, `close`, `volume`.
   - Conversión de la columna `datetime` a índice temporal.

3. **Filtrado**
   - Conserva solo **días hábiles bursátiles** (se eliminan fines de semana y feriados de mercado de EE.UU.).
   - Conversión de marcas de tiempo de **UTC → US/Eastern**.
   - Filtrado de **horario de mercado** (09:30–16:00) más pre-market (desde 08:30).

4. **Validación de registros diarios**
   - Verificación de que cada día contenga la cantidad esperada de registros minuto a minuto.
   - Detección y eliminación de días incompletos o con irregularidades.

5. **Chequeo de continuidad temporal**
   - Confirmación de que los datos intradía estén en intervalos consecutivos de 1 minuto, sin gaps.

6. **Dataset final**
   - Guardado del dataset limpio en formato `.parquet` dentro de Google Drive.

---

**Resultado:** Un dataset intradía del MNQ completamente limpio y estandarizado, listo para la ingeniería de factores y el modelado.

## 0. Configuración del Entorno

### 0.1. Acceso a Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías

In [4]:
import sys
!{sys.executable} -m pip install -q pandas_market_calendars
print("✅ Librería instalada: pandas_market_calendars")

✅ Librería instalada: pandas_market_calendars


### 0.3. Importación de librerías

In [5]:
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import pandas as pd
from tabulate import tabulate

# Calendario de mercados
import pandas_market_calendars as mcal
import pandas as pd
import requests
from io import StringIO


# **1. Contexto y fuente de datos**

Los datos corresponden al contrato MNQ (Micro E-mini Nasdaq 100) descargados desde NinjaTrader con frecuencia de un minuto (formato OHLCV).

- Open: precio de apertura
- High: precio máximo
- Low: precio mínimo
- Close: precio de cierre
- Volume: volumen negociado

Los datos están en la zona horaria UTC.


# **2. Generación de dataset desde archivos históricos**

Dado que los contratos se encuentran almacenados en archivos .txt dentro de la carpeta historicos_mnq, es necesario unificarlos en un único dataset consolidado.

La siguiente función se encarga de leer los archivos .txt, asignar nombres a las columnas correspondientes y establecer la columna datetime como índice temporal del dataframe.

In [10]:
def generar_df ():

    # Ruta a los archivos .txt
    ruta_historicos_drive = f'{drive_path}/data/00_source/*.txt'

    # Determinar qué ruta usar
    if glob.glob(ruta_historicos_drive):
        print("Usando históricos desde Google Drive")
        ruta_historicos = ruta_historicos_drive
    else:
        raise FileNotFoundError("No se encontraron archivos históricos en el Drive.")

    # Lista para almacenar DataFrames individuales
    df_mnq = []

    # Leer todos los archivos .txt
    for archivo in glob.glob(ruta_historicos):
        df = pd.read_csv(
            archivo,
            sep=';',
            header=None,
            names=['datetime', 'open', 'high', 'low', 'close', 'volume'],
            dtype={'open': float, 'high': float, 'low': float, 'close': float, 'volume': int}
        )

        # Convertir columna 'datetime' al formato datetime real
        df['datetime'] = pd.to_datetime(df['datetime'], format='%Y%m%d %H%M%S')

        # Establecer como índice
        df.set_index('datetime', inplace=True)

        df_mnq.append(df)

    # Unir todos los DataFrames
    df_mnq_raw = pd.concat(df_mnq)
    # Ordenar por fecha si es necesario
    df_mnq_raw.sort_index(inplace=True)

    return df_mnq_raw

El siguiente bloque de código verifica si el dataset consolidado ya ha sido generado previamente.

En particular, comprueba la existencia del archivo mnq_raw.parquet.

- Si el archivo está presente, se carga directamente en la variable df_mnq.

- En caso contrario, se invoca la función generate_dataset() para generar el dataset a partir de los archivos originales.

In [15]:
import os
import pandas as pd

def load_or_build_raw_dataset():

    raw_dir = f"{drive_path}/data/01_raw"
    mnq_raw_data_file = f"{raw_dir}/mnq_raw.parquet"

    # Crear carpeta si no existe
    os.makedirs(raw_dir, exist_ok=True)

    if os.path.exists(mnq_raw_data_file):
        print("Archivo encontrado en disco. Cargando dataset local...")
        df_mnq_raw = pd.read_parquet(mnq_raw_data_file)

    else:
        print("Archivo no encontrado. Generando dataset desde archivos históricos...")
        df_mnq_raw = generar_df()
        df_mnq_raw.to_parquet(mnq_raw_data_file, index=True)
        print("Dataset generado y guardado localmente.")

    return df_mnq_raw

In [16]:
df_mnq_raw = load_or_build_raw_dataset()

Archivo no encontrado. Generando dataset desde archivos históricos...
Usando históricos desde Google Drive
Dataset generado y guardado localmente.


In [17]:
df_mnq_raw

,open,high,low,close,volume
datetime,,,,,
2019-12-23 03:01:00,8718.50,8718.75,8718.50,8718.50,9
2019-12-23 03:02:00,8718.25,8718.25,8718.00,8718.25,14
2019-12-23 03:03:00,8718.25,8718.50,8718.00,8718.25,74
2019-12-23 03:04:00,8718.25,8719.00,8718.25,8718.50,10
2019-12-23 03:05:00,8718.50,8719.00,8718.50,8719.00,6
...,...,...,...,...,...
2025-06-15 23:59:00,21687.25,21687.75,21682.75,21686.50,216
2025-06-16 00:00:00,21686.50,21691.75,21686.00,21687.25,103
2025-06-16 00:01:00,21687.25,21694.00,21684.50,21688.00,266
